In [4]:
! pwd 

/Users/Shared


In [5]:
import os
os.chdir("../")

In [6]:
%pwd

'/Users'

In [12]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [13]:
def load_pdf_files(data):
    loader=DirectoryLoader(data, glob="**/*.pdf", show_progress=True, loader_cls=PyPDFLoader)
    documents=loader.load()
    return documents

In [15]:
import os

# Prefer repo-local `data` folder, but fall back to a user-writable location if creation fails.
data_dir = "data"
if not os.path.isdir(data_dir):
    try:
        os.makedirs(data_dir, exist_ok=True)
        print(f"Created '{data_dir}' in repo. Add PDFs into it and re-run the loader if needed.")
    except PermissionError:
        fallback = os.path.expanduser("~/Astha_A_data")
        os.makedirs(fallback, exist_ok=True)
        print(f"Permission denied creating '{data_dir}'. Using fallback: {fallback}\nAdd PDFs into the fallback folder and re-run the loader.")
        data_dir = fallback

extracted_data = load_pdf_files(data_dir)

Permission denied creating 'data'. Using fallback: /Users/imac/Astha_A_data
Add PDFs into the fallback folder and re-run the loader.


0it [00:00, ?it/s]


In [10]:
extracted_data

NameError: name 'extracted_data' is not defined

In [ ]:
len(extracted_data)

In [ ]:
from typing import List
from langchain.schema import Document
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs : List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source", "")
        minimal_doc = Document(
            page_content=doc.page_content,
            metadata={"source": src}
        )
        minimal_docs.append(minimal_doc)
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [ ]:
## split the documents into chunks 
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20,length_function=len)
    docs = text_splitter.create_documents([doc.page_content for doc in minimal_docs])
    return docs

In [ ]:
docs=text_split(minimal_docs)
print(f"Number of chunks: {len(docs)}")

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

In [ ]:
! pip install sentence_transformers

In [ ]:
embedding=download_embeddings()

In [ ]:
embedding

In [ ]:
embedding.embed_query("hello world")

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPEN_AI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
os.environ["OPENAI_API_KEY"] = OPEN_AI_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [ ]:
import pinecone
# Initialize Pinecone client using the API key from environment
pinecone.init(api_key=PINECONE_API_KEY, environment="us-west1-gcp")
# Example: connect to an index (uncomment and set your index name)
# index = pinecone.Index("your-index-name")